# 04 - Actualizar indice de footprints

Esta etapa toma el resultado de la carga al mosaic dataset, exporta las geometrias `FOOTPRINT`, las anexa a la feature class de indice y recalcula `Nombre_de_Vuelo` con la secuencia usada por el flujo PAO.

In [ ]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'core').exists() and (candidate / 'flujo_geosupport_final' / 'settings.json').exists():
            return candidate
        if candidate.name.lower() == 'flujo_geosupport_final' and (candidate / 'settings.json').exists():
            parent = candidate.parent
            if (parent / 'core').exists():
                return parent
    raise RuntimeError('No se pudo resolver PROJECT_ROOT: debe existir core/ y flujo_geosupport_final/settings.json')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import flujo_geosupport_final.scripts.geosupport_flujo_completo as flujo
flujo = importlib.reload(flujo)

FINAL_DIR = PROJECT_ROOT / 'flujo_geosupport_final'
SETTINGS_PATH = FINAL_DIR / 'settings.json'
with SETTINGS_PATH.open('r', encoding='utf-8') as file_obj:
    SETTINGS = json.load(file_obj)
flujo.apply_flow_settings(SETTINGS)

## Parametros

`LOAD_INPUT_CSV` corresponde a `01_load_input_with_attributes.csv` de la etapa 03. `LOAD_RESULTS_CSV` corresponde a `02_mosaic_load_results.csv` de la misma etapa.

In [ ]:
STAGE_03_OUTPUT = FINAL_DIR / 'outputs' / '03_carga_datastore_mosaico'
LOAD_INPUT_CSV = STAGE_03_OUTPUT / '01_load_input_with_attributes.csv'
LOAD_RESULTS_CSV = STAGE_03_OUTPUT / '02_mosaic_load_results.csv'
OUTPUT_DIR = FINAL_DIR / 'outputs' / '04_actualizar_footprints_indice'

APPLY_CHANGES = bool(SETTINGS.get('apply_changes_default', False))

print('Settings:', SETTINGS_PATH)
print('Load input:', LOAD_INPUT_CSV)
print('Load results:', LOAD_RESULTS_CSV)
print('Mosaic dataset:', flujo.PATH_MOSAIC_DATASET)
print('Feature indice:', flujo.PATH_FC_FOOTPRINT_INDICE)
print('Salida:', OUTPUT_DIR)
print('Aplicar cambios:', APPLY_CHANGES)

## Ejecutar actualizacion

In [ ]:
load_df = pd.read_csv(LOAD_INPUT_CSV)
results_df = pd.read_csv(LOAD_RESULTS_CSV)

stage = flujo.update_footprints_stage(
    load_df=load_df,
    results_df=results_df,
    output_dir=OUTPUT_DIR,
    apply_changes=APPLY_CHANGES,
)

display(stage['summary_df'])
display(stage['names_df'].head(20))